# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 8 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'projects page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'discuss page', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'chat page', 'url': 'https://huggingface.co/chat'},
  {'type': 'brand p

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
tencent/HY-MT1.5-1.8B
Updated
3 days ago
•
2.67k
•
508
zai-org/GLM-4.7
Updated
11 days ago
•
31.5k
•
1.42k
MiniMaxAI/MiniMax-M2.1
Updated
7 days ago
•
179k
•
800
Qwen/Qwen-Image-2512
Updated
3 days ago
•
8.3k
•
348
LGAI-EXAONE/K-EXAONE-236B-A23B
Updated
about 8 hours ago
•
921
•
287
Browse 2M+ models
Spaces
Running
Featured
3.44k
Wan2.2 Animate
👁
3.44k
Wan2.2 Animate
Running
on
Zero
975
Z Image Turbo
🖼
975
Generate images from text prompts
Running
on
CPU Upgrade
386
Omni Image Editor
🖼
386
Image edit, text to image, face swap, image 

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ntencent/HY-MT1.5-1.8B\nUpdated\n3 days ago\n•\n2.67k\n•\n508\nzai-org/GLM-4.7\nUpdated\n11 days ago\n•\n31.5k\n•\n1.42k\nMiniMaxAI/MiniMax-M2.1\nUpdated\n7 days ago\n•\n179k\n•\n800\nQwen/Qwen-Image-2512\nUpdated\n3 days ago\n•\n8.3k\n•\n348\nLGAI-EXAONE/K-EXAONE-236B-A23B\nUpdated\nabout 8 hours ago\n•\n921\n•\n287\nBrowse 2M+ models\nSpaces\nRunning\nFeatured\n3.44k\nWan2.2 Animate\n👁\n3.

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the vibrant AI community and collaboration platform shaping the future of machine learning. At its core, Hugging Face Hub serves as a central place where anyone—engineers, scientists, and end users—can share, explore, discover, and experiment with open-source machine learning models, datasets, and applications. 

With a fast-growing global community, widely adopted open-source tools, and a talented science team pushing the boundaries of AI, Hugging Face stands at the heart of the AI revolution fostering an open and ethical AI future.

---

## What Hugging Face Offers

- **Models:** Access and collaborate on over 2 million machine learning models spanning text, images, video, audio, and even 3D modalities.
- **Datasets:** Explore and share from a library of 500,000+ datasets to fuel diverse AI research and development.
- **Spaces:** Host and demo your AI applications in an interactive environment called Spaces to build portfolios and showcase AI creativity.
- **Community:** Join a dynamic, supportive community focused on learning, sharing, and advancing AI responsibly.
- **Documentation & Resources:** Comprehensive docs, blogs, forums, and tutorials available to accelerate learning and adoption.

---

## Enterprise Solutions

Hugging Face provides tailored solutions for teams and organizations seeking enterprise-grade AI development platforms:

- **Team & Enterprise Hub:** Scalable AI platform with features including Single Sign-On, granular access controls, audit logs, and private storage.
- **Security & Compliance:** Enterprise-grade security with control over data regions, token management, and detailed analytics.
- **Advanced Compute:** Access enhanced compute options like ZeroGPU for scaling AI workloads efficiently.
- **Dedicated Support & Flexible Contracts:** Customized plans to meet organizational needs with professional support.

---

## Company Culture

- **Open Collaboration:** Emphasizes open-source values and community-driven innovation.
- **Ethical AI:** Commitment to building an ethical AI future through transparency and inclusivity.
- **Learning & Growth:** Encourages building personal ML portfolios, sharing work publicly, and continual skills development.
- **Diverse & Talented Team:** Home to a passionate science and engineering team advancing cutting-edge AI technologies.

---

## Careers at Hugging Face

Hugging Face is actively hiring and offers a dynamic work environment for individuals passionate about advancing AI technology. Join the team to contribute to open-source projects, collaborate with global innovators, and be part of a fast-moving AI revolution.

(Current openings can be found on their careers page.)

---

## Join the AI Community

- **Explore AI Apps or Browse Models:** Dive into millions of models and applications.
- **Create & Share:** Host your ML models, datasets, and applications publicly for global collaboration.
- **Accelerate Your Machine Learning:** Leverage powerful open-source tools and enterprise compute options.

Sign up today to build, learn, and shape the future with Hugging Face.

---

## Brand & Visual Identity

- **Colors:** Signature palette includes yellow (#FFD21E), orange (#FF9D00), and gray (#6B7280).
- **Logo:** Available in SVG, PNG, and AI formats to represent the brand consistently.

---

## Connect with Hugging Face

- Website: [huggingface.co](https://huggingface.co)
- GitHub, Twitter, LinkedIn, Discord communities for staying engaged with the latest updates and collaborations.

---

**Hugging Face** — Building the future of AI, together.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face - The AI Community Building the Future

---

## About Hugging Face

Hugging Face is the central collaboration platform dedicated to the machine learning (ML) community. It serves as a hub where millions of ML engineers, scientists, and enthusiasts converge to create, share, and explore models, datasets, and applications across all AI modalities—text, image, video, audio, and 3D.

The platform hosts over 2 million models, 500,000+ datasets, and more than a million applications, establishing itself as the heart of the AI revolution. The community and team continuously push the boundaries of open and ethical AI, empowering the next generation of AI developers and users.

---

## Platform Highlights

- **Hugging Face Hub:** A vibrant space for unlimited public hosting and collaboration on ML models, datasets, and applications.
- **Spaces:** An interactive environment for building and sharing AI demos and applications.
- **Open Source Tools:** Accelerate your ML projects with popular libraries and frameworks that are widely used throughout the community.
- **Multi-Modal Support:** Easily explore and create models supporting diverse data types from text and images to video, audio, and 3D.

Build your public ML portfolio, share your projects with the global community, and collaborate on cutting-edge AI research and applications.

---

## Enterprise Solutions

Designed for teams and organizations, Hugging Face offers an advanced AI platform with enterprise-grade security and management features:

- **Secure Access:** Single Sign-On (SSO) integration with identity providers.
- **Granular Permissions:** Manage repositories and resources with detailed access controls and resource groups.
- **Data Governance:** Region selector, audit logs, and token management for full control and compliance.
- **Scalability:** Advanced compute options including ZeroGPU quota boosts for faster performance.
- **Private Features:** Additional private storage, dataset viewers, and controls for confidential projects.
- **Analytics:** Unified dashboard tracking usage, billing, and resource consumption.

Enterprise plans start at $20/user/month, with flexible contracts available for larger scale needs.

---

## Community & Culture

Hugging Face fosters a welcoming, open, and collaborative community culture where transparency, openness, and ethical AI development are core values. The company empowers AI practitioners at every level to learn, build, and contribute by providing tools, resources, and a strong sense of shared purpose.

The culture thrives on:

- **Open Collaboration:** Encouraging public sharing and peer-to-peer knowledge exchange.
- **Innovation:** Supporting cutting-edge research and fast iteration cycles.
- **Inclusion:** Building an accessible platform for diverse users worldwide.
- **Ethics:** Promoting responsible AI through open-source practices and ethical guidelines.

---

## Careers

Join a fast-growing, mission-driven company at the forefront of AI innovation. Hugging Face offers exciting opportunities for:

- Machine Learning Engineers and Scientists
- Open Source Contributors
- Product and Platform Developers
- Data Scientists
- Community and Developer Relations Specialists

Working at Hugging Face means contributing to a meaningful mission of shaping the future of AI with openness and collaboration. The team benefits from a creative, supportive environment rich with learning opportunities.

---

## Why Choose Hugging Face?

- Access to the largest open ML model and dataset repository.
- Collaborative tools designed specifically for machine learning workflows.
- A growing ecosystem of applications enabling real-world AI impact.
- Enterprise-ready solutions for secure, scalable AI deployments.
- A passionate team committed to ethical and inclusive AI.

---

## Connect & Discover

Explore models, datasets, and AI applications at:  
[https://huggingface.co](https://huggingface.co)

Join the community via GitHub, Twitter, LinkedIn, Discord, and more to stay updated and contribute to the AI future.

---

**Colors & Brand Assets**  
Hugging Face’s cheerful brand is represented with signature colors:  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

Use their open brand assets to align with the company’s vibrant and inclusive identity.

---

*Hugging Face – Building the future of AI together.*

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 14 relevant links


# Welcome to Hugging Face – The AI Community Building the Future!

---

## Who Are We?  
Imagine a place where machine learning wizards, data sorcerers, and AI alchemists gather to share their spells — uh, models — datasets, and apps. That’s Hugging Face! We’re *the* platform where the AI community collaborates, creates, and sometimes even has a little fun while building the future.

Our motto? **"Keep it open. Keep it ethical. Keep it hugging."** 💛

---

## What’s Cooking in the AI Kitchen?

- **1 Million+ Models** — From image generators to language wizards, our treasure trove of open-source ML models grows faster than you can say "neural network."  
- **250,000+ Datasets** — Feeding AI brains with everything from chat prompts to persona profiles. Hungry for data? Dig in!  
- **400,000+ Applications & Spaces** — Launch apps, share your ML portfolio, or just show off cool demos that make your friends say, “Whoa, AI can do that?”  
- **Multimodal Madness** — Text, image, video, audio, even 3D...if AI had a Swiss Army knife, we’d be it.  

---

## Customers & Community  
Whether you’re a student trying to get your AI feet wet, a startup looking to scale your genius, or an enterprise aiming to deploy heavy-duty models in the real world, Hugging Face has your back.

With the fastest growing community of *machine learning enthusiasts* and the support of some seriously big names and organizations, here’s a place where:

- **Freelancers** can build a portfolio and get noticed.  
- **Researchers** can push boundaries openly and ethically.  
- **Businesses** can accelerate AI adoption with our paid Compute and Enterprise suites.  

Join 1.29k+ Spaces and thousands more running models that power everything from video generation to AI-powered image editing.

---

## Culture & Career – Geek Out with Us!  
We believe collaboration beats isolation every day. Our culture?

- Open source at heart ❤️  
- Ethical AI advocates  
- Casual tea-drinkers and serious problem solvers  
- Always learning, always sharing, always growing  

Want to build machine learning tools that millions will use? Hugging Face is where your skills meet endless possibilities. From ML engineers to community managers, our doors are wide open (virtual hugs included).

---

## Speed Up Your AI Journey  
No need to code in the dark alone or fight for GPU time — deploy models and apps with a few clicks on optimized inference endpoints, starting at just $0.60/hour for GPU!

Whether you want to host that killer new model or just tweak an existing one, we give you the tools and community support to **move faster, build smarter, and hug tighter**.

---

## Quick Hugging Face Facts  
- **Founded:** Around the corner from the future  
- **Colors:** Bright yellow (#FFD21E), orange (#FF9D00), and sleek gray (#6B7280) — because AI should be as vibrant as its ideas!  
- **Mascot:** Friendly face with a warm smile (because AIs could learn a thing or two about friendliness here)  

---

## Ready to Join the AI Hug Circle?  

Sign up, share your work, explore millions of models and datasets, and get your AI career (or project!) hugging new heights.

[Explore AI Apps](#) | [Browse 1M+ Models](#) | [Sign Up & Join The Fun](#)

---

*Hugging Face — where the future of AI isn’t just created; it’s hugged into existence.* 🤗✨

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>